# Омни-ассистент: экскурсия голосом персонажа

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtakoy/llm-engineer-OMNI-assistant-dz/blob/main/notebooks/omni_gradio.ipynb)

Ноутбук работает в двух средах.

**Локально.** Ядро - `.venv` проекта: пакет `assistant` ставится туда через
`uv sync`. Модели раздаёт ollama, поднятая на своей машине.

**В Colab.** Нужна среда с видеокартой: Среда выполнения → Сменить среду
выполнения → T4 GPU. Клетки сами склонируют репозиторий, поставят пакет и
поднимут ollama.

Порядок работы в интерфейсе: загрузить фотографию персонажа, задать вопрос
текстом или голосом, нажать кнопку. Описание персонажа, текст лекции и
проигрыватели с озвучкой появляются по мере готовности.

In [1]:
import sys

IS_COLAB = "google.colab" in sys.modules

print(f"среда: {'colab' if IS_COLAB else 'локальная'}")
print(f"питон: {sys.version.split()[0]}")

среда: локальная
питон: 3.13.9


In [ ]:
if IS_COLAB:
    # ВОЗМОЖНО, что после первой попытки придется перезапустить сеанс - это ок!

    !nvidia-smi
    !git clone https://github.com/samtakoy/llm-engineer-OMNI-assistant-dz.git /content/omni
    # Ставим правкой на месте: PROJECT_ROOT в variables.py считается от файла
    # пакета, и при обычной установке каталоги проекта уехали бы в site-packages.
    !pip install -e /content/omni
else:
    print("локальная среда: пакет стоит в .venv, установка пропущена")

локальная среда: пакет стоит в .venv, установка пропущена


In [ ]:
import json
import subprocess
import time
import urllib.error
import urllib.request

# Имена моделей отсюда уезжают в переменные окружения следующей клетки.
# Бюджет видеокарты T4: 4b в четырёхбитном кванте около 3 ГБ, зрение 4b около
# 3.5 ГБ, распознавание речи около 1.5 ГБ, синтез около 0.2 ГБ.
TEXT_MODEL = "qwen3.5:4b"
VISION_MODEL_NAME = "qwen3-vl:4b"

OLLAMA_URL = "http://127.0.0.1:11434"


def wait_for_ollama(seconds: int) -> bool:
    """
    Ждёт, пока сервер ollama начнёт отвечать.

    Аргументы:
        seconds: сколько ждать.

    Возвращает:
        True, если сервер ответил за отведённое время.
    """
    deadline = time.monotonic() + seconds

    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout = 2):
                return True
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            time.sleep(2)

    return False


def installed_models() -> set[str]:
    """
    Спрашивает у ollama список загруженных моделей.

    Возвращает:
        Имена моделей вместе с тегами.
    """
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout = 5) as answer:
        listing = json.load(answer)

    return {item["name"] for item in listing.get("models", [])}


if IS_COLAB:
    # Установщик ollama распаковывает архив zstd, в образе Colab его нет.
    !apt-get install -y -qq zstd
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(
        ["ollama", "serve"],
        stdout = subprocess.DEVNULL,
        stderr = subprocess.DEVNULL,
    )

if not wait_for_ollama(seconds = 60):
    raise RuntimeError(f"ollama не отвечает на {OLLAMA_URL}: подними сервер и повтори клетку")

present = installed_models()
missing = [name for name in (TEXT_MODEL, VISION_MODEL_NAME) if name not in present]

if not missing:
    print(f"модели на месте: {TEXT_MODEL}, {VISION_MODEL_NAME}")
elif IS_COLAB:
    for name in missing:
        !ollama pull {name}
else:
    # Гигабайты на чужую машину без спроса не тянем.
    print("не хватает моделей, скачай и повтори клетку:")
    for name in missing:
        print(f"    ollama pull {name}")

модели на месте: qwen3.5:4b, qwen3-vl:4b


In [ ]:
import os

# Переменные окружения
os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = TEXT_MODEL
os.environ["VISION_PROVIDER"] = "ollama"
os.environ["VISION_MODEL"] = VISION_MODEL_NAME

In [ ]:
from assistant.webui import launch_app

# Запуск
launch_app(is_inline = False, is_shared = IS_COLAB)

/Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-23-omni-assistant/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[speech] загрузка модели medium (auto, int8)
[вопрос] распознано: Ты гид по городу Пятигорска, расскажи мне и до детей 10 лет экскурсию по горе Машук.
[облик]
Три женщины стоят в ряд перед ярким огненным фоном, их позы уверенные и вызывающие.  
Первая слева с рыжими волосами в черном костюме, вторая — блондинка в черном топе и джинсах, третья — с черными волосами в черной блузке и узорчатых брюках. У всех светлая кожа, серьезные лица, смотрят прямо в камеру.  
Первая выражает уверенность, брови прямые; вторая улыбается широко, демонстрируя вызов; третья сохраняет напряженное выражение.  
Обстановка наполнена динамикой: огненный фон, яркий свет, красный и оранжевый цвета создают атмосферу опасности и привлекательности.  
Надпись «Прекрасные и опасные» подчеркивает агрессивную привлекательность персонажей.  
Кто это: Charlie's Angels (American film)
[рассказчик] Чарлис Энджелс (женский)
характер: Уверенные, дерзкие и опасные женщины, готовые к любой авантюре.
обращение: Девчонки, подруги

Using cache found in /Users/samtakot/devs/learnings/llm-eng/homework/llm-eng-23-omni-assistant/.cache/silero/snakers4_silero-models_master
<torch_package_0>.multi_acc_v3_package.py:286: SyntaxWarning: invalid escape sequence '\^'
  text = re.sub(r'[^{}]'.format(self.symbols[3:] + '\^'), '', text)


[голос] baya, темп fast, высота high, эффект cartoon (medium)
[голос] заголовки читает eugene
[разметка] кусок 1
<break time="300ms"/>Огоо!, девчонки, подруги, смертные дамы и охотницы за приключениями! Вы готовы к удару? Я, ваш дерзкий гид Чарлис Энджелс, готова превратить этот сухой список фактов в настоящий драйв для вашего города Пятигорска! Мы не просто идем на гору Машук; мы идем туда, где сила и красота встречаются с историей! Погнали!
[speaking] в синтез (разметка и настройки голоса): <speak><prosody rate="110%" pitch="high"><p><s><break time="300ms"/>Огоо!, девчонки, подруги, смертные дамы и охотницы за приключениями!</s><s>Вы готовы к удару?</s><s>Я, ваш дерзкий гид Чарлис Энджелс, готова превратить этот сухой список фактов в настоящий драйв для вашего города Пятигорска!</s><s>Мы не просто идем на гору Машук; мы идем туда, где сила и красота встречаются с историей!</s><s>Погнали!</s></p></prosody></speak>
[озвучка] кусок 1: /Users/samtakot/devs/learnings/llm-eng/homework/llm-